In [ ]:
import ROOT
import math
# import pandas as pd

# print(f"ROOT version: {ROOT.__version__}")
import numpy as np
from scipy.special import j0
import plotly.graph_objects as go
from scipy.integrate import fixed_quad


In [ ]:
def read_data_file(filename):
    """Read data file and return arrays for x, y, y_error"""
    x_vals = []
    y_vals = []
    y_errs = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 3:
                    x_vals.append(float(parts[0]))
                    y_vals.append(float(parts[1]))
                    y_errs.append(float(parts[2]))
    
    return x_vals, y_vals, y_errs

In [ ]:
# Load experimental data for all energies from ATLAS
x_atlas_all, y_atlas_all, yerr_atlas_all = read_data_file('../../../data/ens_atlas_difc0_2.dat')

# Function to process data for each energy block
def process_data(x_data, y_data, yerr_data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        if end is None:
            end = len(x_data)
        x_values.append(x_data[start:end])
        y_values.append(y_data[start:end])
        y_errors.append(yerr_data[start:end])
    
    return x_values, y_values, y_errors


In [ ]:
#ranges for each energy 
atlas_blocks = [(0, 29), (29, 58), (58, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(x_atlas_all, y_atlas_all, yerr_atlas_all, atlas_blocks)

# Extract values by energy
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [ ]:
# defining parameters/constants
b_0 = (33 - 6) / (12 * np.pi)
lambda_qcd = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25


#ensemble parameters
param_mg_atlas_pl = 0.421
param_eps_atlas_pl = 0.0753
param_a1_atlas_pl = 1.517
param_a2_atlas_pl = 2.05

In [ ]:
#--------------------------------------
# Eq 22 - GE
#--------------------------------------
def m2_pl(q2, mg):
    lambda2 = lambda_qcd ** 2
    rho_mg_2 = rho * (mg ** 2)
    ratio = math.log((q2 + rho_mg_2) / lambda2) / math.log(rho_mg_2 / lambda2)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

#--------------------------------------
# Eq 26 - GE
#--------------------------------------
def G_p(q2, a1, a2):
    t = -q2
    return np.exp(-(a1 * np.abs(t) + a2 * np.abs(t) ** 2))


#--------------------------------------
# Eq 24 - GE
#--------------------------------------
def alpha_D(q2, mg, m2_type):
    m2_func = m2_type(q2, mg)
    return 1.0 / (b_0 * (q2 + m2_func) * math.log((q2 + 4 * m2_func) / (lambda_qcd ** 2)))

#--------------------------------------
# Eq 7 - GE
#--------------------------------------
def T_1(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    G0 = G_p(q, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2


#--------------------------------------
# Eq 8 - GE
#--------------------------------------
def T_2(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


#--------------------------------------
# Eq 11 - GE
#--------------------------------------
def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

#--------------------------------------
# Eq 6 - GE
#--------------------------------------
def born_amp(diff_T, s, eps, t):
    alpha_pomeron = 1.0 + eps + 0.25 * t
    s_tilde = s/s0

    return 1j * s * 8 * (s_tilde**(alpha_pomeron - 1)) * diff_T

In [ ]:
def root_1d_integrator(func, lower_limit, upper_limit):
    
    """
    PURPOSE: Perform one-dimensional numerical integration using ROOT's 
    adaptive integration method.

    PARAMETERS:

        func (callable): Function to be integrated. Must accept a single 
            floating-point argument.

        lower_limit (float): Lower bound of the integration interval.

        upper_limit (float): Upper bound of the integration interval.

    RETURNS:
        tuple: Tuple containing:
            - [0] (float): Estimated value of the integral.
            - [1] (float): Estimated uncertainty of the integral.
    """

    # creating Functor to be used by ROOT's integrator
    functor = ROOT.Math.Functor1D(func)

    type = ROOT.Math.IntegrationOneDim.kADAPTIVE   # integration type
    absTol = 1e-4
    relTol = 1e-4
    size   = 20
    rule   = ROOT.Math.Integration.kGAUSS15   
        
    # defining Integrator object
    integrator = ROOT.Math.IntegratorOneDim(type, absTol, relTol, size, rule)
    integrator.SetFunction(functor)
    
    # calculating integral and error 
    result = integrator.Integral(lower_limit, upper_limit)
    error = integrator.Error()
    
    return result, error


In [ ]:
#--------------------------------------
# Eq 7 and Eq 8 - GE
#--------------------------------------

def k_integral(k, mg, a1, a2, m2_func, q):
    """
    PURPOSE: Define the integrand function over φ (phi) for a fixed k value.

    PARAMETERS:

        k (float): Momentum magnitude variable for the outer integral.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

    RETURNS:
        callable: Integrand function of φ (phi), defined as:
            f(φ) = k * [T₁(k, φ, mg, a1, a2, m2_func, q) - T₂(k, φ, mg, a1, a2, m2_func, q)].
    """
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    return integrand


def phi_integral(phi, mg, a1, a2, m2_func, q, k_max):
    """
    PURPOSE: Compute the inner integral over k for a fixed φ (phi) value 
    using ROOT's 1D integrator.

    PARAMETERS:

        phi (float): Azimuthal angle variable (in radians).

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

        k_max (float): Upper limit for the k integration.

    RETURNS:
        float: Estimated value of the k-integral for the given φ (phi):
            ∫₀^{k_max} k [T₁(k, φ) - T₂(k, φ)] dk.
    """
    def inner_in_k(k):
        return k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                    T_2(k, phi, mg, a1, a2, m2_func, q))
    
    result, _ = root_1d_integrator(inner_in_k, 0, k_max)
    return result


def compute_k_phi_integral(mg, a1, a2, m2_func, q, k_max):
    """
    PURPOSE: Compute the full two-dimensional integral over k and φ (phi),
    corresponding to Eqs. (7) and (8) in the GE model.

    PARAMETERS:

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

        k_max (float): Upper limit for the k integration.

    RETURNS:
        tuple: Tuple containing:
            - [0] (float): Estimated value of the double integral:
                ∫₀^{2π} ∫₀^{k_max} k [T₁(k, φ) - T₂(k, φ)] dk dφ
            - [1] (float): Estimated uncertainty from the outer φ integration.
    """
    result, error = root_1d_integrator(
        lambda phi: phi_integral(phi, mg, a1, a2, m2_func, q, k_max),
        0,
        2 * math.pi
    )
    return result, error



# TESTING 

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0
k_max = 13000

compute_k_phi_integral(mg, a1, a2, m2_pl, q, k_max)

In [ ]:
#--------------------------------------  
#   COMPUTING SIGMA TOT BORN
#--------------------------------------


#start parameters
start_sqrt_s = 100
max_sqrt_s = 13000
step_size = 100

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0

lst_sigma_tot_born_list = []
lst_sqrt_s_list = []

# start initial sqrt for while loop  
current_sqrt_s = start_sqrt_s

while current_sqrt_s <= max_sqrt_s:

    current_s = current_sqrt_s**2

    # calculates diff t (eq 7 and 8)
    diff_t,_ = compute_k_phi_integral(mg, a1, a2, m2_pl, q, current_sqrt_s)

    # calculating born amplitude
    born_amp_value = born_amp(diff_t, current_s, param_eps_atlas_pl, 0)
    # print(born_amp_value)

    #calculating sigma tot born
    born_sigma_tot_value = born_sigma_tot(born_amp_value, current_s)
    # print(born_sigma_tot_value)

    # append results to list
    lst_sigma_tot_born_list.append(born_sigma_tot_value)
    lst_sqrt_s_list.append(current_sqrt_s)

    # increase step
    current_sqrt_s += step_size 
    print(born_sigma_tot_value)

In [ ]:
#--------------------------------------
# Eq 23 - EIK
#--------------------------------------


def chi_eikonal(s, b, eps, mg, a1, a2, m2_func, born_amp_func):
    """
    PURPOSE: Compute the complex eikonal function χ(s, b) as defined in Eq. (23),
    using the Born amplitude and the nested integral over k and φ.

    PARAMETERS:

        s (float): Mandelstam variable s (squared center-of-mass energy).

        b (float): Impact parameter in femtometers (fm) or GeV⁻¹, depending on model units.

        eps (float): Model parameter ε controlling energy dependence of the amplitude.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer q.

        born_amp_func (callable): Function that computes the complex Born amplitude:
            A_Born(diff_T, s, eps, t), where t = -q² and diff_T is obtained from the 
            nested integral over k and φ.

    RETURNS:
        complex: Complex value of the eikonal function χ(s, b), computed as:
            (1 / s) × [∫ Re(A_Born) dq  +  i ∫ Im(A_Born) dq].

    NOTES:

        - Implements Eq. (23) from the generalized eikonal (GE) formalism.
        - Uses `root_1d_integrator` for numerical evaluation of the q-integral.
        - The integration limits (0 → 0.2) are set empirically and may depend 
          on the energy range or chosen model normalization.
    """

    def integrand_real(q):
        t = -q**2  
        
        # Compute diff_T for this q value
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)
        )

        # Compute Born amplitude
        amp_born = born_amp_func(diff_T, s, eps, t)
        
        # Integrand for the real part
        return q * j0(b * q) * amp_born.real
    
    def integrand_imag(q):
        t = -q**2
        
        # Compute diff_T for this q value
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)
        )

        # Compute Born amplitude
        amp_born = born_amp_func(diff_T, s, eps, t)
        
        # Integrand for the imaginary part
        return q * j0(b * q) * amp_born.imag
    
    # Perform numerical integration for real and imaginary components
    real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, 0.2)
    imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, 0.2)

    # Combine real and imaginary parts
    integral_result = real_integral_result + 1j * imag_integral_result

    return integral_result / s


print(chi_eikonal(7000**2, 10.0, param_eps_atlas_pl, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl, born_amp))

In [ ]:
#--------------------------------------
# Eq 24 - EIK (using the new chi_eikonal)
#--------------------------------------

def eik_amp(s, t, eps, mg, a1, a2, m2_func, born_amp_func, q_max=0.2):
    """
    PURPOSE: Compute the eikonalized scattering amplitude A_eik(s, t) 
    as defined in Eq. (24) using the eikonal function χ(s, b).

    PARAMETERS:

        s (float): Mandelstam variable s (squared center-of-mass energy).

        t (float): Mandelstam variable t (momentum transfer squared, typically negative).

        eps (float): Model parameter ε controlling the energy dependence of the amplitude.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        born_amp_func (callable): Function that computes the complex Born amplitude:
            A_Born(diff_T, s, eps, t), where diff_T is obtained from the nested integral 
            over k and φ.

        q_max (float, optional): Maximum momentum transfer q used internally in χ(s, b) 
            integration. Default is 0.2.

    RETURNS:
        complex: Complex eikonalized amplitude A_eik(s, t), given by:
            i s × ∫₀^{b_max} b J₀(b√(-t)) [1 - exp(iχ(s, b))] db.

    NOTES:

        - Implements Eq. (24) from the generalized eikonal (GE) formalism.
        - Uses the `chi_eikonal` function to evaluate χ(s, b) at each b value.
        - The integration is performed over b ∈ [0, 30], which may be adjusted 
          depending on the physical range or model normalization.
        - `root_1d_integrator` is used for adaptive numerical integration.
    """

    q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
    def integrand_real(b_val):
        # Compute eikonal function χ(s, b)
        chi_val = chi_eikonal(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func)
        
        # Compute [1 - exp(iχ(s, b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s, b))]
        return (b_val * j0(b_val * q) * one_minus_exp).real
    
    def integrand_imag(b_val):
        # Compute eikonal function χ(s, b)
        chi_val = chi_eikonal(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func)
        
        # Compute [1 - exp(iχ(s, b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s, b))]
        return (b_val * j0(b_val * q) * one_minus_exp).imag
    
    # Integrate real and imaginary parts separately
    real_integral, _ = root_1d_integrator(integrand_real, 0.0, 30)
    imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, 30)
    
    # Combine real and imaginary components
    integral_result = real_integral + 1j * imag_integral
    
    # Final amplitude: i s times the integral
    return 1j * s * integral_result

print(eik_amp(7000**2, 0,param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl, born_amp))

In [57]:
#  \file
#  \ingroup tutorial_fit
#  \notebook -nodraw
#  Example on how to use the new Minimizer class in ROOT
#   Show usage with all the possible minimizers.
#  Minimize a 4D function
#
#  input : minimizer name + algorithm name
#  randomSeed: = <0 : fixed value: 0 random with seed 0; >0 random with given seed
#
#  \macro_code
#
#  \author Lorenzo Moneta

from math import e
from os import error
import ROOT
import numpy as np
import math


def Function4D(vecx):
    x = vecx[0]
    y = vecx[1]
    z = vecx[2]
    w = vecx[3]
    
    # 4D Rastrigin function
    A = 10
    return (A * 4 + 
            (x**2 - A * math.cos(2 * math.pi * x)) +
            (y**2 - A * math.cos(2 * math.pi * y)) +
            (z**2 - A * math.cos(2 * math.pi * z)) +
            (w**2 - A * math.cos(2 * math.pi * w)))

# create minimizer giving a name and a name (optionally) for the specific algorithm
#  possible choices are:
#     minimizerName                  algoName
#
#     Minuit                     Migrad, Simplex,Combined,Scan  (default is Migrad)
#     Minuit2                    Migrad, BFGS, Simplex,Combined,Scan  (default is Migrad)
#     GSLMultiMin                ConjugateFR, ConjugatePR, BFGS, BFGS2, SteepestDescent
#     GSLSimAn
#     Genetic


def root_minimize(func, minimizerName,
                          algoName,
                          maxCalls,
                          maxIterations,
                          tolerance,
                          printLevel,
                          precision):

    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if (not minimizer):
        raise RuntimeError(
            "Cannot create minimizer \"{}\". Maybe the required library was not built?".format(minimizerName))

    # Set tolerance and other minimizer parameters, one can also use default
    # values

    minimizer.SetMaxFunctionCalls(maxCalls)  # working for Minuit/Minuit2
    # for GSL minimizers - no effect in Minuit/Minuit2
    minimizer.SetPrecision(precision)
    minimizer.SetMaxIterations(maxIterations)
    minimizer.SetTolerance(tolerance)
    minimizer.SetStrategy(2) 
    minimizer.SetPrintLevel(printLevel)

    # Create function wrapper for minimizer

    f = ROOT.Math.Functor(func, 4)

    # Starting point
    variable = [-0.03, 0.05, 0.05, 0.08]
    step = [0.1, 0.1, 0.1, 0.1]

    minimizer.SetFunction(f)

    # Set the free variables to be minimized !
    minimizer.SetVariable(0, "x", variable[0], step[0])
    minimizer.SetVariable(1, "y", variable[1], step[1])
    minimizer.SetVariable(2, "z", variable[2], step[2])
    minimizer.SetVariable(3, "w", variable[3], step[3])

    # Do the minimization
    ret = minimizer.Minimize()
    minimizer.Hesse()

    edm = minimizer.Edm()
    hesse = minimizer.Hesse()
    if hesse:
        print("✓ Hesse calculation successful")
        cov_status = minimizer.CovMatrixStatus()
    
        if cov_status == 3:
            # Extract Hesse errors
            errors = [minimizer.Errors()[i] for i in range(len(variable))]
            print(f"Valid Hesse errors: {errors}")
        else:
            print(f"✗ Covariance matrix status = {cov_status} (need 3 for valid errors)")
    else:
        print("✗ Hesse calculation failed")


    xs = minimizer.X()
    errors = minimizer.Errors()

    print(f'''
        Minimizer {minimizerName} - {algoName} converged to:
        x = {xs[0]} +- {errors[0]}
        y = {xs[1]} +- {errors[1]}
        z = {xs[2]} +- {errors[2]}
        w = {xs[3]} +- {errors[3]}

        Edm = {edm}        
        ''')

    # Real minimum is f(xmin) = 0
    if (ret and minimizer.MinValue() < 1E-4):
        print("Minimizer {} - {} converged to the right minimum!".format(minimizerName, algoName))
    else:
        print("Minimizer {} - {} failed to converge !!!".format(minimizerName, algoName))
        raise RuntimeError("NumericalMinimization failed to converge!")


if __name__ == "__main__":
    root_minimize(Function4D, minimizerName="Minuit2",
                          algoName="Migrad",
                          maxCalls=10000,
                          maxIterations=10000,
                          tolerance=0.01,
                          printLevel=0,
                          precision=1E-8)


✓ Hesse calculation successful
Valid Hesse errors: [0.07099664415217198, 0.0709966452839694, 0.07099664528305515, 0.07099664469270149]

        Minimizer Minuit2 - Migrad converged to:
        x = -2.5922880762110502e-06 +- 0.07099664415217198
        y = 1.0167353004188624e-05 +- 0.0709966452839694
        z = 1.0167353004517476e-05 +- 0.07099664528305515
        w = -4.4634817573407405e-06 +- 0.07099664469270149

        Edm = 4.6303300533001654e-08        
        
Minimizer Minuit2 - Migrad converged to the right minimum!


In [37]:
import ROOT
import numpy as np


def model_function(x, params):
    """Simple linear model: y = a + b*x"""
    return params[0] + params[1] * x


def chi_square(params, x_data, y_data, y_errors):
    """
    Chi-square: measures goodness of fit
    χ² = Σ [(y_measured - y_model) / σ]²
    """
    chi2 = 0.0
    for i in range(len(x_data)):
        y_model = model_function(x_data[i], params)
        y_measured = y_data[i]
        sigma = y_errors[i]
        
        residual = (y_measured - y_model) / sigma
        chi2 += residual * residual
    
    return chi2


def fit_data_get_real_errors(x_data, y_data, y_errors):
    """
    Fit data and extract statistical errors (like in papers)
    """
    
    # Create minimizer
    minimizer = ROOT.Math.Factory.CreateMinimizer("Minuit2", "Migrad")
    minimizer.SetMaxFunctionCalls(100000)
    minimizer.SetTolerance(1e-6)
    minimizer.SetPrintLevel(1)
    
    # Wrap chi-square function
    def chi2_func(params):
        return chi_square(params, x_data, y_data, y_errors)
    
    f = ROOT.Math.Functor(chi2_func, 2)  # 2 parameters
    minimizer.SetFunction(f)
    
    # Set initial parameter values
    minimizer.SetVariable(0, "a", 0.0, 0.1)  # intercept
    minimizer.SetVariable(1, "b", 1.0, 0.1)  # slope
    
    # Run minimization
    print("\nFitting data...")
    success = minimizer.Minimize()
    
    if not success:
        print("❌ Fit failed!")
        return None
    
    # Extract results
    params = [minimizer.X()[0], minimizer.X()[1]]
    errors = [minimizer.Errors()[0], minimizer.Errors()[1]]
    chi2_min = minimizer.MinValue()
    ndof = len(x_data) - 2  # degrees of freedom
    chi2_reduced = chi2_min / ndof
    
    # Check covariance matrix quality
    cov_status = minimizer.CovMatrixStatus()
    
    print(f"\n{'='*60}")
    print(f"FIT RESULTS")
    print(f"{'='*60}")
    print(f"Status: {'✓ SUCCESS' if success else '✗ FAILED'}")
    print(f"Covariance matrix: {'✓ VALID' if cov_status == 3 else '✗ INVALID'}")
    print(f"χ²/ndf = {chi2_min:.2f} / {ndof} = {chi2_reduced:.3f}")
    print(f"\nFitted parameters (STATISTICAL ERRORS):")
    print(f"  a (intercept) = {params[0]:.4f} ± {errors[0]:.4f}")
    print(f"  b (slope)     = {params[1]:.4f} ± {errors[1]:.4f}")
    
    # Covariance matrix
    cov_aa = minimizer.CovMatrix(0, 0)
    cov_ab = minimizer.CovMatrix(0, 1)
    cov_bb = minimizer.CovMatrix(1, 1)
    
    print(f"\nCovariance matrix:")
    print(f"  [[{cov_aa:9.6f}, {cov_ab:9.6f}]")
    print(f"   [{cov_ab:9.6f}, {cov_bb:9.6f}]]")
    
    # Correlation coefficient
    rho = cov_ab / (errors[0] * errors[1])
    print(f"\nCorrelation: ρ(a,b) = {rho:.3f}")
    
    return params, errors, chi2_reduced


if __name__ == "__main__":
    
    print("="*60)
    print("SIMPLE EXAMPLE: LINEAR FIT WITH REAL ERRORS")
    print("="*60)
    
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # TRUE parameters (what we're trying to recover)
    true_a = 5.0  # intercept
    true_b = 2.0  # slope
    
    print(f"\nTrue model: y = {true_a} + {true_b}*x")
    
    # Generate synthetic data
    n_points = 20
    x_data = np.linspace(0, 10, n_points)
    
    # Generate y values with noise
    y_true = true_a + true_b * x_data
    y_errors = np.ones(n_points) * 1.0  # constant error σ = 1.0
    y_data = y_true + np.random.normal(0, 1.0, n_points)  # add Gaussian noise
    
    print(f"\nGenerated {n_points} data points with σ = 1.0")
    
    # Fit the data
    result = fit_data_get_real_errors(
        x_data.tolist(),
        y_data.tolist(),
        y_errors.tolist()
    )
    
    if result:
        params, errors, chi2_red = result
        
        print(f"\n{'='*60}")
        print(f"COMPARISON WITH TRUE VALUES")
        print(f"{'='*60}")
        
        # Calculate "pull" (deviation in units of error)
        pull_a = (params[0] - true_a) / errors[0]
        pull_b = (params[1] - true_b) / errors[1]
        
        print(f"Parameter a:")
        print(f"  Fitted: {params[0]:.4f} ± {errors[0]:.4f}")
        print(f"  True:   {true_a:.4f}")
        print(f"  Pull:   {pull_a:.2f}σ {'✓' if abs(pull_a) < 2 else '⚠'}")
        
        print(f"\nParameter b:")
        print(f"  Fitted: {params[1]:.4f} ± {errors[1]:.4f}")
        print(f"  True:   {true_b:.4f}")
        print(f"  Pull:   {pull_b:.2f}σ {'✓' if abs(pull_b) < 2 else '⚠'}")
        
        print(f"\n{'='*60}")
        print(f"INTERPRETATION")
        print(f"{'='*60}")
        print(f"✓ χ²/ndf ≈ 1: Good fit (errors correctly estimated)")
        print(f"✓ |pull| < 2σ: Fitted values consistent with truth")
        print(f"✓ These are the errors you report in papers!")
        print(f"\nIn a paper, you'd write:")
        print(f"  a = {params[0]:.2f} ± {errors[0]:.2f}")
        print(f"  b = {params[1]:.2f} ± {errors[1]:.2f}")

SIMPLE EXAMPLE: LINEAR FIT WITH REAL ERRORS

True model: y = 5.0 + 2.0*x

Generated 20 data points with σ = 1.0

Fitting data...

FIT RESULTS
Status: ✓ SUCCESS
Covariance matrix: ✓ VALID
χ²/ndf = 10.92 / 18 = 0.607

Fitted parameters (STATISTICAL ERRORS):
  a (intercept) = 5.7746 ± 0.4309
  b (slope)     = 1.8108 ± 0.0737

Covariance matrix:
  [[ 0.185714, -0.027143]
   [-0.027143,  0.005429]]

Correlation: ρ(a,b) = -0.855

COMPARISON WITH TRUE VALUES
Parameter a:
  Fitted: 5.7746 ± 0.4309
  True:   5.0000
  Pull:   1.80σ ✓

Parameter b:
  Fitted: 1.8108 ± 0.0737
  True:   2.0000
  Pull:   -2.57σ ⚠

INTERPRETATION
✓ χ²/ndf ≈ 1: Good fit (errors correctly estimated)
✓ |pull| < 2σ: Fitted values consistent with truth
✓ These are the errors you report in papers!

In a paper, you'd write:
  a = 5.77 ± 0.43
  b = 1.81 ± 0.07
Minuit2Minimizer: Minimize with max-calls 100000 convergence for edm < 1e-06 strategy 1
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = 10.9191981239138425
Edm   

In [ ]:
# #--------------------------------------
# # Eq 23 - EIK
# #--------------------------------------

# q_max = 0.2

# def chi_eikonal(s, b, eps, mg, a1, a2, m2_func):
#     """
#     Eikonal function χ(s,b) from eq. (23)
#     χ(s,b) = (1/s) ∫ q dq J₀(bq) A_Born(s,t)
#     where t = -q²
#     """
#     def integrand_real(q):
#         t = -q**2  
        
#         # Calculate diff_T for this q value        
#         diff_T, _ = compute_k_phi_integral(
#             mg=mg,
#             a1=a1,
#             a2=a2,
#             m2_func=m2_func,
#             q=q,
#             k_max=np.sqrt(s)  # Use sqrt(s) as k_max
#         )

#         # Calculate Born amplitude for this t
#         amp_born = born_amp(diff_T, s, eps, t)
        
#         # Integrand: q * J₀(b*q) * A_Born(s,t)
#         return q * j0(b * q) * amp_born.real
    
#     def integrand_imag(q):
#         t = -q**2  # t = -q² as specified
        
#         # Calculate diff_T for this q value        
#         diff_T, _ = compute_k_phi_integral(
#             mg=mg,
#             a1=a1,
#             a2=a2,
#             m2_func=m2_func,
#             q=q,
#             k_max=np.sqrt(s)  # Use sqrt(s) as k_max
#         )

#         # Calculate Born amplitude for this t
#         amp_born = born_amp(diff_T, s, eps, t)
        
#         # Integrand: q * J₀(b*q) * A_Born(s,t)
#         return q * j0(b * q) * amp_born.imag
    
#     real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, q_max)  
#     imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, q_max)  

#     integral_result = real_integral_result + 1j*imag_integral_result

#     return integral_result / s

# print(chi_eikonal(7000**2, 10.0, param_eps_atlas_pl, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl))

# #--------------------------------------
# # Eq 24 - EIK
# #--------------------------------------

# b_max = 30  # Maximum impact parameter

# def eikonal_amplitude(s, t, eps, mg, a1, a2, m2_func):
#     """
#     Eikonalized amplitude from eq. (24)
#     A_eik(s,t) = i s ∫ b db J₀(b√(-t)) [1 - exp(iχ(s,b))]
#     where t is the Mandelstam variable (negative)
#     """
#     q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
#     def integrand_real(b):
#         # Calculate eikonal function χ(s,b)
#         chi_val = chi_eikonal(s, b, eps, mg, a1, a2, m2_func)
        
#         # Compute [1 - exp(iχ(s,b))]
#         exp_term = np.exp(1j * chi_val)
#         one_minus_exp = 1.0 - exp_term
        
#         # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
#         return (b * j0(b * 0) * one_minus_exp).real
    
#     def integrand_imag(b):
#         # Calculate eikonal function χ(s,b)
#         chi_val = chi_eikonal(s, b, eps, mg, a1, a2, m2_func)
        
#         # Compute [1 - exp(iχ(s,b))]
#         exp_term = np.exp(1j * chi_val)
#         one_minus_exp = 1.0 - exp_term
        
#         # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
#         return (b * j0(b * 0) * one_minus_exp).imag
    
#     # Integrate real and imaginary parts separately
#     real_integral, _ = root_1d_integrator(integrand_real, 0.0, b_max)
#     imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, b_max)
    
#     integral_result = real_integral + 1j * imag_integral
    
#     # Final amplitude: i s times the integral
#     return 1j * s * integral_result

# print(eikonal_amplitude(7000**2,-0.04,param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl))